In [1]:
!pip install -q -U "protobuf<4"
!pip install -q -U bitsandbytes peft accelerate datasets transformers sacrebleu evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, bu

In [2]:
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"

import torch

if torch.cuda.device_count() > 1:
    raise RuntimeError(
        "LỖI: Máy vẫn đang nhận diện 2 GPU! "
        "Bạn hãy chọn 'Run' > 'Factory reset' và chạy lại từ đầu."
    )
print(f"✅ Hệ thống đã nhận diện chính xác: {torch.cuda.device_count()} GPU (OK để chạy 4-bit).")

from datasets import Dataset
import re
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

class config:
    data_dir = "/kaggle/input/vlsp-medical-data" 
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    output_dir = "/kaggle/working/qwen_mt_en_vi"
    lora_r = 16
    lora_alpha = 32
    lora_dropout = 0.05
    num_epochs = 3
    batch_size = 4
    learning_rate = 2e-4
    
def load_data(src_filename, trg_filename):
    src_path, trg_path = "", ""
    for root, dirs, files in os.walk(config.data_dir):
        if src_filename in files: src_path = os.path.join(root, src_filename)
        if trg_filename in files: trg_path = os.path.join(root, trg_filename)
            
    if not src_path or not trg_path: raise FileNotFoundError("Không tìm thấy file!")

    with open(src_path, 'r', encoding='utf-8') as f: src_lines = [l.strip() for l in f]
    with open(trg_path, 'r', encoding='utf-8') as f: trg_lines = [l.strip() for l in f]
    return list(zip(src_lines, trg_lines))

def clean_text(text):
    text = ' '.join(text.split())
    return re.sub(r'\s+([.,!?;:])', r'\1', text).strip()

def create_instruction_format(src, trg, direction="en-vi"):
    if direction == "en-vi":
        sys = "You are a professional medical translator specialized in translating English medical texts to Vietnamese. Preserve all medical terminology, drug names, and clinical context accurately."
        user = f"Translate this English medical text to Vietnamese:\n\n{src}"
    else:
        sys = "You are a professional medical translator..."
        user = f"Translate this Vietnamese medical text to English:\n\n{src}"

    return {"messages": [{"role": "system", "content": sys}, {"role": "user", "content": user}, {"role": "assistant", "content": trg}]}

print("Đang load dữ liệu...")
data_raw = load_data("train.en.txt", "train.vi.txt")
data_raw = data_raw[:50000] 

train_raw, val_raw = train_test_split(data_raw, test_size=0.1, random_state=42)

train_dataset = Dataset.from_list([create_instruction_format(clean_text(s), clean_text(t)) for s, t in train_raw if s and t])
val_dataset = Dataset.from_list([create_instruction_format(clean_text(s), clean_text(t)) for s, t in val_raw if s and t])

tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

def format_chat_template(examples):
    texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in examples["messages"]]
    tokenized = tokenizer(texts, truncation=True, max_length=256, padding="max_length", return_tensors="pt")
    tokenized["labels"] = tokenized["input_ids"].clone()
    tokenized["labels"][tokenized["labels"] == tokenizer.pad_token_id] = -100
    return tokenized

train_dataset = train_dataset.map(format_chat_template, batched=True)
val_dataset = val_dataset.map(format_chat_template, batched=True)

print("Đang load Model 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_use_double_quant=False
)

model = AutoModelForCausalLM.from_pretrained(
    config.model_name, 
    quantization_config=bnb_config, 
    device_map="auto", 
    trust_remote_code=True
)

model.config.use_cache = False 
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=config.lora_r, lora_alpha=config.lora_alpha, lora_dropout=config.lora_dropout,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    gradient_accumulation_steps=2,
    learning_rate=config.learning_rate,
    fp16=True,
    
    logging_steps=200,     
    eval_strategy="steps",
    eval_steps=1000,        

    save_strategy="epoch",  
    save_total_limit=1,     
    report_to="none",
    
    gradient_checkpointing=True, 
    gradient_checkpointing_kwargs={"use_reentrant": False} 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("🚀 Bắt đầu training...")
trainer.train()

trainer.save_model(os.path.join(config.output_dir, "final_model"))
tokenizer.save_pretrained(os.path.join(config.output_dir, "final_model"))
print("Xong!")

✅ Hệ thống đã nhận diện chính xác: 1 GPU (OK để chạy 4-bit).


2025-12-16 17:55:56.804993: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765907757.023346      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765907757.087442      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Đang load dữ liệu...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
🚀 Bắt đầu training...


Step,Training Loss,Validation Loss
1000,1.143600,1.144080
2000,1.095200,1.090321
3000,1.060900,1.061266
4000,1.023600,1.037333
5000,1.031400,1.019182
6000,0.917700,1.010674
7000,0.907500,1.004005
8000,0.925100,0.993090
9000,0.879800,0.986739
10000,0.899800,0.978623


Xong!
